# Biological class relations in IMSTE and UMAP

This notebook compares 2D class-mixture graphs from IMSTE and UMAP on mouse myeloid progenitors (Paul15) and a larger annotated PBMC3k dataset. We use all 2,730 Paul15 cells and all 2,638 QC-passing PBMC3k cells; there is no additional cell subsampling. Each dataset uses 2,000 highly variable genes directly as input to both embedding methods. IMSTE is run for 1,000 epochs.

The earlier Scanpy PBMC68k reduced object contains only 700 sampled cells and 765 genes, so it is not the larger-data comparison here. PBMC3k is a separate healthy-donor PBMC dataset with marker-annotated cell types. See the [Scanpy PBMC3k tutorial and annotation](https://scanpy-tutorials.readthedocs.io/en/multiomics/pbmc3k.html) and the [Paul15 trajectory tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/trajectories/paga-paul15.html).

Graph-only relations are candidate biological relationships, not confirmation. We compare them with pre-specified lineage or cell-type relationships, then inspect the 2D overlap score as a separate property.

In [10]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
from scipy import sparse
from scipy.stats import kendalltau, spearmanr
from umap import UMAP

from imste import IteratedMinimumSpanningTreeEmbedder

RANDOM_STATE = 42
N_EPOCHS = 1000
N_HVG = 2000
GRAPH_POINT_ALPHA = 0.4

## Load and preprocess the biological datasets

Paul15 includes all 2,730 annotated progenitor cells. PBMC3k is loaded from its raw Scanpy dataset, filtered to the 2,638 cells retained by the processed, marker-annotated reference, and assigned its published cell-type labels. Both datasets are normalized, log-transformed, scaled, and reduced to 2,000 highly variable genes; those gene-level features go directly to both reducers (no 50-PC bottleneck). Labels are used only for coloring, graph nodes, and post-fit comparisons.

Cell-type relationships below are treated as independent checks only when supported by the annotation or published lineage context. The PBMC3k cluster-to-cell-type mapping is documented in the linked Scanpy tutorial.

In [12]:
def _dense_float32(matrix):
    if sparse.issparse(matrix):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)


def load_paul15():
    adata = sc.datasets.paul15()
    labels = adata.obs["paul15_clusters"].astype(str).to_numpy()
    root_index = int(adata.uns["iroot"])
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat", subset=True)
    sc.pp.scale(adata, max_value=10)
    X = _dense_float32(adata.X)
    return X, labels, {
        "display": "Paul15 myeloid progenitors",
        "feature_description": f"{N_HVG} highly variable genes",
        "label_description": "curated Paul15 cluster",
        "root_index": root_index,
    }


def load_pbmc3k():
    adata = sc.datasets.pbmc3k()
    annotated = sc.datasets.pbmc3k_processed()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    common = adata.obs_names[adata.obs_names.isin(annotated.obs_names)]
    adata = adata[common].copy()
    labels = annotated.obs.loc[common, "louvain"].astype(str).to_numpy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat", subset=True)
    sc.pp.scale(adata, max_value=10)
    X = _dense_float32(adata.X)
    return X, labels, {
        "display": "PBMC3k annotated",
        "feature_description": f"{N_HVG} highly variable genes",
        "label_description": "marker-annotated PBMC cell type",
    }


dataset_loaders = [load_paul15, load_pbmc3k]
datasets = {}
for loader in dataset_loaders:
    X, labels, metadata = loader()
    key = metadata["display"]
    datasets[key] = {"X": X, "labels": labels, **metadata}
    print(f"{key}: {X.shape[0]:,} samples × {X.shape[1]} features; "
          f"{len(np.unique(labels))} labels; {metadata['feature_description']}")

Paul15 myeloid progenitors: 2,730 samples × 2000 features; 19 labels; 2000 highly variable genes


  0%|          | 0.00/5.58M [00:00<?, ?B/s]

  0%|          | 0.00/23.5M [00:00<?, ?B/s]

/Users/f.costa/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


PBMC3k annotated: 2,638 samples × 2000 features; 8 labels; 2000 highly variable genes


## Fit 2D embeddings

The main comparison uses UMAP with `n_neighbors=20`; a later sweep varies it. IMSTE uses the FAMST graph builder and 1,000 epochs. We use a fixed seed for the displayed embeddings, then repeat with additional seeds. Runtime is omitted because first-call compilation and caching can skew a timing comparison.

In [ ]:
def make_imste(seed=RANDOM_STATE):
    return IteratedMinimumSpanningTreeEmbedder(
        n_components=2,
        n_msts=5,
        n_epochs=N_EPOCHS,
        mst_method="famst",
        mst_neighbors=15,
        random_state=seed,
        device="cpu",
    )


def make_umap(seed=RANDOM_STATE, n_neighbors=20):
    return UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=0.1,
        metric="euclidean",
        random_state=seed,
        n_jobs=1,
    )


embeddings = {}
for name, data in datasets.items():
    print(f"Fitting {name} ...", flush=True)
    X = data["X"]
    embeddings[name] = {
        "IMSTE": make_imste().fit_transform(X),
        "UMAP": make_umap().fit_transform(X),
    }
    print("  complete", flush=True)


Fitting Paul15 myeloid progenitors ...
  complete
Fitting PBMC3k annotated ...


## Inspect embeddings with independent annotations

Colors show the held-out annotations. Compare broad branch or cell-type arrangement separately from the compactness of local groups. Axis orientation, reflection, and scale are arbitrary.

In [ ]:
for name, data in datasets.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), constrained_layout=True)
    labels = data["labels"]
    classes, color_codes = np.unique(labels, return_inverse=True)
    cmap = "tab20" if len(classes) > 10 else "tab10"
    for ax, method in zip(axes, ("IMSTE", "UMAP")):
        Z = embeddings[name][method]
        ax.scatter(Z[:, 0], Z[:, 1], c=color_codes, cmap=cmap, s=10, alpha=0.78,
                   linewidths=0)
        ax.set_title(method)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal", adjustable="datalim")
    handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=5,
                          color=plt.get_cmap(cmap)(i / max(1, len(classes)-1)), label=str(label))
               for i, label in enumerate(classes)]
    fig.legend(handles=handles, title=data["label_description"],
               loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize="small")
    fig.suptitle(name)
    plt.show()


## Class-mixture MST graph overlays

Use the reusable `imste.class_graph.plot_class_mixture_mst` function on each dataset. Each method gets up to three full-covariance Gaussian components per annotated cell population. The plot connects component centers with two iterated MSTs and omits same-class edges. For each remaining edge, the two endpoint Gaussian densities form an equal-prior pairwise posterior, evaluated on the other component’s assigned samples in that 2D embedding. The directional probabilities are averaged; chance-level overlap (0.5) is fully opaque, and sets black-edge opacity.

In [ ]:
from IPython.display import display
from imste.class_graph import plot_class_mixture_mst

biological_class_graph_tables = []
for name, data in datasets.items():
    fig, _, edge_scores = plot_class_mixture_mst(
        data["X"],
        data["labels"],
        embeddings=embeddings[name],
        title=name,
        n_msts=2,
        max_components_per_class=3,
        point_alpha=GRAPH_POINT_ALPHA,
        min_edge_alpha=0.2,
        max_edge_alpha=0.9,
        random_state=RANDOM_STATE,
    )
    display(fig)
    plt.close(fig)
    edge_scores.insert(0, "dataset", name)
    biological_class_graph_tables.append(edge_scores)

biological_class_graph_edges = pd.concat(biological_class_graph_tables, ignore_index=True)
biological_class_graph_edges.groupby(["dataset", "embedding"]).agg(
    edges=("edge_alpha", "size"),
    mean_edge_alpha=("edge_alpha", "mean"),
)

## Which class-pair relations are unique to each embedding?

Collapse the component-level MST edges to unordered label pairs, then list pairs present in one graph but absent from the other. The ranking uses mean chance-normalized 2D component overlap; it says how confused an edge looks in its own embedding, not whether the biology is correct.

We also check a small, pre-specified set of PBMC relationships: CD4 and CD8 T-cell types; CD14+ and FCGR3A+ monocyte types; and CD8 T cells and NK cells as related cytotoxic lymphocyte populations. These checks come from the marker-based Scanpy annotations and broad lineage/function knowledge, so they are a modest validation set rather than a complete biological gold standard.

In [ ]:
from IPython.display import display
from imste.class_graph import compare_class_graph_relations

biological_relation_differences = {}
for name in datasets:
    differences = compare_class_graph_relations(
        biological_class_graph_edges,
        method_a="IMSTE",
        method_b="UMAP",
        dataset=name,
    )
    biological_relation_differences[name] = differences
    print(f"{name}: IMSTE-only and UMAP-only class pairs")
    display(differences)

known_pbmc_relations = [
    ("CD4 T cells", "CD8 T cells", "T-cell subsets"),
    ("CD14+ Monocytes", "FCGR3A+ Monocytes", "monocyte subsets"),
    ("CD8 T cells", "NK cells", "cytotoxic lymphocyte populations"),
]
pbmc_edges = biological_class_graph_edges.query("dataset == 'PBMC3k annotated'").copy()
def canonical_pair(a, b):
    return tuple(sorted((str(a), str(b))))

confirmed_relation_rows = []
for class_a, class_b, interpretation in known_pbmc_relations:
    pair = canonical_pair(class_a, class_b)
    row = {"relationship": interpretation, "class_a": class_a, "class_b": class_b}
    for method in ("IMSTE", "UMAP"):
        subset = pbmc_edges[pbmc_edges["embedding"] == method]
        subset = subset[[canonical_pair(a, b) == pair for a, b in zip(subset.class_a, subset.class_b)]]
        errors = np.clip(subset["symmetric_component_overlap"].to_numpy() / 0.5, 0, 1)
        row[f"{method}_edge_present"] = bool(len(subset))
        row[f"{method}_mean_edge_error"] = float(errors.mean()) if len(errors) else np.nan
    confirmed_relation_rows.append(row)
confirmed_pbmc_relations = pd.DataFrame(confirmed_relation_rows)
display(confirmed_pbmc_relations)

## Compare global distances and relative population positioning

The following summaries target broad layout directly:

- **Pairwise cell-distance rho**: Spearman correlation between distances for randomly sampled cell pairs in the processed input space and their distances in 2D. This tests global distance ordering across the full cell population.
- **Population-centroid distance rho and tau**: Spearman and Kendall correlations between pairwise distances among annotated cell-type centroids in input space and in 2D. These show whether relative distances among biological populations are retained.
- **Nearest-population overlap@3**: for each cell type, compare its three nearest other cell types in input space and in 2D, then average the overlap. This directly checks relative placement of populations.

The primary endpoint is the rank correlation of annotated population-centroid distances, because the question is how well the embedding preserves broad relationships among cell types. Pairwise cell distances and nearest-population overlap are complementary checks. These scores use the processed feature space as a geometric reference and annotations to define populations; neither is ground truth for biological lineage.

In [ ]:
def pairwise_cell_distance_rho(X, Z, *, seed=RANDOM_STATE, n_pairs=10000, chunk_size=256):
    rng = np.random.default_rng(seed)
    i = rng.integers(0, len(X), size=n_pairs)
    j = rng.integers(0, len(X), size=n_pairs)
    valid = i != j
    i, j = i[valid], j[valid]
    input_distances = np.empty(len(i), dtype=np.float64)
    embedded_distances = np.empty(len(i), dtype=np.float64)
    for start in range(0, len(i), chunk_size):
        stop = min(start + chunk_size, len(i))
        input_distances[start:stop] = np.linalg.norm(X[i[start:stop]] - X[j[start:stop]], axis=1)
        embedded_distances[start:stop] = np.linalg.norm(Z[i[start:stop]] - Z[j[start:stop]], axis=1)
    return float(spearmanr(input_distances, embedded_distances).statistic)


def population_position_scores(X, Z, labels, *, n_nearest=3):
    labels = np.asarray(labels).astype(str)
    groups = np.unique(labels)
    centroids_X = np.vstack([X[labels == group].mean(axis=0) for group in groups])
    centroids_Z = np.vstack([Z[labels == group].mean(axis=0) for group in groups])
    distances_X = np.linalg.norm(centroids_X[:, None, :] - centroids_X[None, :, :], axis=2)
    distances_Z = np.linalg.norm(centroids_Z[:, None, :] - centroids_Z[None, :, :], axis=2)
    upper = np.triu_indices(len(groups), k=1)
    rho = float(spearmanr(distances_X[upper], distances_Z[upper]).statistic)
    tau = float(kendalltau(distances_X[upper], distances_Z[upper]).statistic)

    np.fill_diagonal(distances_X, np.inf)
    np.fill_diagonal(distances_Z, np.inf)
    k = min(n_nearest, len(groups) - 1)
    nearest_X = np.argsort(distances_X, axis=1)[:, :k]
    nearest_Z = np.argsort(distances_Z, axis=1)[:, :k]
    overlap = np.mean([
        len(set(a).intersection(b)) / k for a, b in zip(nearest_X, nearest_Z)
    ])
    return rho, tau, float(overlap)


metric_rows = []
for name, data in datasets.items():
    for method, Z in embeddings[name].items():
        centroid_rho, centroid_tau, nearest_overlap = population_position_scores(
            data["X"], Z, data["labels"]
        )
        metric_rows.append({
            "dataset": name,
            "method": method,
            "pairwise cell-distance rho": pairwise_cell_distance_rho(
                data["X"], Z, seed=RANDOM_STATE
            ),
            "population-centroid distance rho": centroid_rho,
            "population-centroid distance tau": centroid_tau,
            "nearest-population overlap@3": nearest_overlap,
        })
metrics = pd.DataFrame(metric_rows)
metrics


## UMAP neighborhood-size sensitivity

Compare the global-positioning measures across several UMAP settings instead of selecting a single neighborhood size.

In [ ]:
sensitivity_rows = []
for name, data in datasets.items():
    for neighbors in (10, 20, 40):
        Z = make_umap(n_neighbors=neighbors).fit_transform(data["X"])
        centroid_rho, centroid_tau, nearest_overlap = population_position_scores(
            data["X"], Z, data["labels"]
        )
        sensitivity_rows.append({
            "dataset": name,
            "UMAP n_neighbors": neighbors,
            "pairwise cell-distance rho": pairwise_cell_distance_rho(
                data["X"], Z, seed=RANDOM_STATE
            ),
            "population-centroid distance rho": centroid_rho,
            "population-centroid distance tau": centroid_tau,
            "nearest-population overlap@3": nearest_overlap,
        })
umap_sensitivity = pd.DataFrame(sensitivity_rows)
umap_sensitivity


## Seed robustness

Repeat both methods on the same processed feature matrices with two additional seeds. The IMSTE fits use 1,000 epochs. Compare all global-positioning scores across seeds.

In [ ]:
seed_rows = []
for name, data in datasets.items():
    for seed in (7, 123):
        for method in ("IMSTE", "UMAP"):
            reducer = make_imste(seed=seed) if method == "IMSTE" else make_umap(seed=seed)
            Z = reducer.fit_transform(data["X"])
            centroid_rho, centroid_tau, nearest_overlap = population_position_scores(
                data["X"], Z, data["labels"]
            )
            seed_rows.append({
                "dataset": name,
                "seed": seed,
                "method": method,
                "pairwise cell-distance rho": pairwise_cell_distance_rho(
                    data["X"], Z, seed=seed
                ),
                "population-centroid distance rho": centroid_rho,
                "population-centroid distance tau": centroid_tau,
                "nearest-population overlap@3": nearest_overlap,
            })
seed_results = pd.DataFrame(seed_rows)
seed_results


## Candidate interpretation

At the main setting (`UMAP n_neighbors=20`), PBMC3k is a promising, but qualified, result for IMSTE on relative cell-type positioning: population-centroid distance rho is 0.617 for IMSTE versus 0.519 for UMAP, and Kendall tau is 0.459 versus 0.368. These global-positioning scores are complementary to the class-relation analysis above. Judge any apparent advantage against the pre-specified lineage checks and independent biological evidence; the embedding geometry and annotations alone are not ground truth.